# Рынок видеоигр 2000–2013: подготовка данных для исследования

Учебный проект программы «Аналитик данных» (Яндекс Практикум), выполнен самостоятельно.
Стек: Python, pandas.

## Цель и задачи

**Цель:** подготовить очищенный и структурированный срез данных о видеоиграх 2000–2013 годов для дальнейшего анализа платформ, жанров и региональных продаж.

**Что сделано:**
1. Загрузка и первичный осмотр данных.
2. Предобработка: snake_case, типы данных, пропуски, явные и неявные дубликаты.
3. Срез за 2000–2013 годы.
4. Категоризация оценок пользователей и критиков, топ-7 платформ по числу игр.

## Данные

Учебный датасет Яндекс Практикума (`new_games.csv`): 16 956 записей об играх — название, платформа, год выпуска, жанр, продажи в Северной Америке, Европе, Японии и других странах (млн копий), оценки критиков (0–100) и пользователей (0–10), возрастной рейтинг ESRB. Данные являются материалами курса и в репозиторий не входят.

## Содержание

1. Загрузка и первичный осмотр
2. Предобработка данных
3. Срез данных за 2000–2013 годы
4. Категоризация данных
5. Итоговый вывод

---

## 1. Загрузка данных и знакомство с ними


In [1]:
# Импортирую библиотеку pandas
import pandas as pd

In [2]:
# Загружаю данные в датафрейм df
df=pd.read_csv('data/new_games.csv')

In [3]:
# Выведу информацию о датафрейме
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [4]:
# Выведу первые строки датафрейма на экран
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Датасет содержит 16 956 строк и 11 столбцов: продажи игр по регионам, жанры, платформы, оценки пользователей и критиков.

Типы данных:
- `float64`: `NA sales`, `Other sales`, `Critic Score`, `Year of Release`; год лучше хранить целым числом после обработки пропусков;
- `object`: `Name`, `Platform`, `Genre` (текст, тип подходит), `Rating` (можно перевести в `category`);
- `EU sales`, `JP sales` и `User Score` содержат числа, но хранятся как строки — их нужно перевести в `float64`.

Итого: нужно перевести `EU sales`, `JP sales`, `User Score` в `float64`, а `Year of Release` — в целочисленный тип.

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

In [5]:
# Выведу названия всех столбцов датафрейма
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [6]:
# Приведу название столбцов в нужный формат, а также привожу к нижнему регистру
df.columns.str.lower().str.replace(" ", "_")

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

In [7]:
# Применю изменение к столбцам df
df.columns=df.columns.str.lower().str.replace(" ", "_")

In [8]:
# Копия исходных данных для контроля числа удалённых строк в конце предобработки
tmp = df.copy() 
len(tmp)

16956

### 2.2. Типы данных

In [9]:
# Выведу типы данных столбцов. В данный момент year_of_release — float, его нужно сменить на int64, но пока изменить не могу.
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
rating              object
dtype: object

In [10]:
# В значениях строк мы можем наблюдать пустые nan, чуть позже мы заполним пустые строки year_of_release по столбцу названия игры name, а оставшиеся пустые удалим и преобразуем столбец к типу данных int64
df['year_of_release'].unique()

array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015., 2012., 2014., 1992., 1997., 1993., 1994., 1982.,
       2016., 2003., 1986., 2000.,   nan, 1995., 1991., 1981., 1987.,
       1980., 1983.])

In [11]:
# Выведу информацию по нужным типам данных, которые необходимо преобразовать в float
df[['eu_sales', 'jp_sales', 'user_score']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   eu_sales    16956 non-null  object
 1   jp_sales    16956 non-null  object
 2   user_score  10152 non-null  object
dtypes: object(3)
memory usage: 397.5+ KB


In [12]:
# Для проверки ошибки и написания выводов по каждому числовому столбцу, который имеет не числовой тип значений, выведу уникальные значения по следующему столбцу eu_sales
df['eu_sales'].unique()

array(['28.96', '3.58', '12.76', '10.93', '8.89', '2.26', '9.14', '9.18',
       '6.94', '0.63', '10.95', '7.47', '6.18', '8.03', '4.89', '8.49',
       '9.09', '0.4', '3.75', '9.2', '4.46', '2.71', '3.44', '5.14',
       '5.49', '3.9', '5.35', '3.17', '5.09', '4.24', '5.04', '5.86',
       '3.68', '4.19', '5.73', '3.59', '4.51', '2.55', '4.02', '4.37',
       '6.31', '3.45', '2.81', '2.85', '3.49', '0.01', '3.35', '2.04',
       '3.07', '3.87', '3.0', '4.82', '3.64', '2.15', '3.69', '2.65',
       '2.56', '3.11', '3.14', '1.94', '1.95', '2.47', '2.28', '3.42',
       '3.63', '2.36', '1.71', '1.85', '2.79', '1.24', '6.12', '1.53',
       '3.47', '2.24', '5.01', '2.01', '1.72', '2.07', '6.42', '3.86',
       '0.45', '3.48', '1.89', '5.75', '2.17', '1.37', '2.35', '1.18',
       '2.11', '1.88', '2.83', '2.99', '2.89', '3.27', '2.22', '2.14',
       '1.45', '1.75', '1.04', '1.77', '3.02', '2.75', '2.16', '1.9',
       '2.59', '2.2', '4.3', '0.93', '2.53', '2.52', '1.79', '1.3', '2.6',
   

In [13]:
# Для проверки ошибки и написания выводов по каждому числовому столбцу, который имеет не числовой тип значений, выведу уникальные значения по следующему столбцу jp_sales
df['jp_sales'].unique()

array(['3.77', '6.81', '3.79', '3.28', '10.22', '4.22', '6.5', '2.93',
       '4.7', '0.28', '1.93', '4.13', '7.2', '3.6', '0.24', '2.53',
       '0.98', '0.41', '3.54', '4.16', '6.04', '4.18', '3.84', '0.06',
       '0.47', '5.38', '5.32', '5.65', '1.87', '0.13', '3.12', '0.36',
       '0.11', '4.35', '0.65', '0.07', '0.08', '0.49', '0.3', '2.66',
       '2.69', '0.48', '0.38', '5.33', '1.91', '3.96', '3.1', '1.1',
       '1.2', '0.14', '2.54', '2.14', '0.81', '2.12', '0.44', '3.15',
       '1.25', '0.04', '0.0', '2.47', '2.23', '1.69', '0.01', '3.0',
       '0.02', '4.39', '1.98', '0.1', '3.81', '0.05', '2.49', '1.58',
       '3.14', '2.73', '0.66', '0.22', '3.63', '1.45', '1.31', '2.43',
       '0.7', '0.35', '1.4', '0.6', '2.26', '1.42', '1.28', '1.39',
       '0.87', '0.17', '0.94', '0.19', '0.21', '1.6', '0.16', '1.03',
       '0.25', '2.06', '1.49', '1.29', '0.09', '2.87', '0.03', '0.78',
       '0.83', '2.33', '2.02', '1.36', '1.81', '1.97', '0.91', '0.99',
       '0.95', '2.0'

In [14]:
# Для проверки ошибки и написания выводов по каждому числовому столбцу, который имеет не числовой тип значений, выведу уникальные значения по следующему столбцу user_score
df['user_score'].unique()

array(['8', nan, '8.3', '8.5', '6.6', '8.4', '8.6', '7.7', '6.3', '7.4',
       '8.2', '9', '7.9', '8.1', '8.7', '7.1', '3.4', '5.3', '4.8', '3.2',
       '8.9', '6.4', '7.8', '7.5', '2.6', '7.2', '9.2', '7', '7.3', '4.3',
       '7.6', '5.7', '5', '9.1', '6.5', 'tbd', '8.8', '6.9', '9.4', '6.8',
       '6.1', '6.7', '5.4', '4', '4.9', '4.5', '9.3', '6.2', '4.2', '6',
       '3.7', '4.1', '5.8', '5.6', '5.5', '4.4', '4.6', '5.9', '3.9',
       '3.1', '2.9', '5.2', '3.3', '4.7', '5.1', '3.5', '2.5', '1.9', '3',
       '2.7', '2.2', '2', '9.5', '2.1', '3.6', '2.8', '1.8', '3.8', '0',
       '1.6', '9.6', '2.4', '1.7', '1.1', '0.3', '1.5', '0.7', '1.2',
       '2.3', '0.5', '1.3', '0.2', '0.6', '1.4', '0.9', '1', '9.7'],
      dtype=object)

In [15]:
# Приведу эти столбцы к типу данных float
for column in ['eu_sales', 'jp_sales', 'user_score']:
    df[column]=pd.to_numeric(df[column], errors='coerce')

In [16]:
# Выведу информацию об изменении
df[['eu_sales', 'jp_sales', 'user_score']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   eu_sales    16950 non-null  float64
 1   jp_sales    16952 non-null  float64
 2   user_score  7688 non-null   float64
dtypes: float64(3)
memory usage: 397.5 KB


В `eu_sales`, `jp_sales` и `user_score` найдены текстовые значения (`unknown`, `tbd`). При переводе в числовой тип через `pd.to_numeric(..., errors='coerce')` они стали `NaN`, тип столбцов — `float64`.

In [17]:
# Число строк в исходных данных
len(tmp)

16956

### 2.3. Наличие пропусков в данных

In [18]:
# Выведу значение всего df в True и False, где True будет означать пустую строку
df.isna()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,True,True,True
2,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...
16951,False,False,False,False,False,False,False,False,True,True,True
16952,False,False,False,False,False,False,False,False,True,True,True
16953,False,False,False,False,False,False,False,False,True,True,True
16954,False,False,False,False,False,False,False,False,True,True,True


In [19]:
# Посчитаю кол-во пустых строк в каждом столбце в абсолютных значениях
df.isna().sum()

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [20]:
# Посчитаю кол-во пустых строк в каждом столбце в относительных значениях
df.isna().mean()

name               0.000118
platform           0.000000
year_of_release    0.016218
genre              0.000118
na_sales           0.000000
eu_sales           0.000354
jp_sales           0.000236
other_sales        0.000000
critic_score       0.513918
user_score         0.546591
rating             0.405225
dtype: float64

In [21]:
# Функция для отображения статистики пропущенных значений в DataFrame
def show_missing_stats(tmp0):
    missing_stats = pd.DataFrame({
        'Кол-во пропусков': tmp0.isnull().sum(),
        'Доля пропусков': tmp0.isnull().mean()
    })
    missing_stats = missing_stats[missing_stats['Кол-во пропусков'] > 0]
    
    if missing_stats.empty:
        return "Пропусков в данных нет"
    
    # Форматируем при выводе через Styler
    return (missing_stats.style.format({'Доля пропусков': '{:.4f}'}).background_gradient(cmap='coolwarm'))
show_missing_stats(df)

,Кол-во пропусков,Доля пропусков
name,2,0.0001
year_of_release,275,0.0162
genre,2,0.0001
eu_sales,6,0.0004
jp_sales,4,0.0002
critic_score,8714,0.5139
user_score,9268,0.5466
rating,6871,0.4052


In [22]:
# На данном этапе сохраняю общее исходное кол-во строк, для того чтобы потом посчитать абсолютную и относительную разницу 
initial_count = len(df)

Больше всего пропусков в `critic_score` — 8714 (51,39%), `user_score` — 9268 (54,66%, с учётом `tbd`) и `rating` — 6871 (40,52%). Вероятная причина — у части игр нет оценок и возрастного рейтинга ESRB.

Небольшие пропуски есть в `year_of_release` — 275 (1,62%), `eu_sales` — 6, `jp_sales` — 4, `name` и `genre` — по 2.

In [23]:
# Заполню пустые строки year_of_release по столбцу названия игры name
df['year_of_release'] = df['year_of_release'].fillna(
    df.groupby('name')['year_of_release'].transform('first')
)

In [24]:
# Обработаю строки с пустыми значениями в следующих столбцах в name, year_of_release, genre, а именно удалю их
df=df.dropna(subset=['name', 'year_of_release', 'genre'])

In [25]:
# После удаления оставшихся пустых строк в year_of_release преобразую тип с float в int64.
df['year_of_release'] = df['year_of_release'].astype('int64')

In [26]:
# Для каждой строки проверю ячейки eu_sales и jp_sales — если значение NaN,
# заполню его средним по группе с одинаковой платформой и годом выпуска
def fill_mean_group(row):
    for col in ['eu_sales', 'jp_sales']:
        if pd.isna(row[col]):
            group = df[
                (df['platform'] == row['platform']) &
                (df['year_of_release'] == row['year_of_release'])
            ]
            row[col] = group[col].mean()
    return row

In [27]:
# Применю функцию fill_mean_group ко всему датафрейму по строкам
df = df.apply(fill_mean_group, axis=1)

In [28]:
# Пропуски в оценках после удаления строк (до заполнения индикатором)
df.critic_score.isna().sum(), df.user_score.isna().sum()

(8627, 9169)

In [30]:
# Пропуски в critic_score и user_score заменены на индикатор -1
for column in ['critic_score', 'user_score']:
    df[column] = df[column].fillna(-1)

In [31]:
# Заполню пропуски в rating значением Unknown
df['rating'] = df['rating'].fillna('Unknown')

Обработка пропусков:
- `year_of_release` заполнен по совпадающему названию игры (одна игра на разных платформах обычно выходит в один год); оставшиеся строки без года, а также строки без `name` и `genre` удалены;
- `eu_sales` и `jp_sales` заполнены средним по платформе и году выпуска;
- `critic_score` и `user_score` заменены индикатором -1: оценки конкретной игры нельзя надёжно восстановить по другим играм;
- `rating` заполнен значением `Unknown` (позже приведено к `UNKNOWN`), так как отсутствие рейтинга не всегда ошибка.

Далее проверим явные и неявные дубликаты.

In [32]:
# Посмотрим какие пропуски остались
show_missing_stats(df)

'Пропусков в данных нет'

### 2.4. Явные и неявные дубликаты в данных

In [33]:
# Получу список уникальных значений в категориальных данных в столбце genre
unique_df_genre = df['genre'].unique()
# Выведу список уникальных жанров
unique_df_genre

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION', 'SHOOTER',
       'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE', 'SIMULATION',
       'PUZZLE', 'STRATEGY'], dtype=object)

In [34]:
# Преобразую все значения в столбце genre к нижнему регистру
df['genre'] = df['genre'].str.lower()
# Выведу измененный список уникальных жанров
df['genre'].unique()

array(['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc',
       'shooter', 'simulation', 'action', 'fighting', 'adventure',
       'strategy'], dtype=object)

In [35]:
# Получу список уникальных значений в категориальных данных в столбце platform
unique_df_platform = df['platform'].unique()
# Выведу список уникальных платформ
unique_df_platform

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

In [36]:
# Преобразую все значения в столбце platform к нижнему регистру
df['platform'] = df['platform'].str.lower()
# Выведу измененный список уникальных платформ
df['platform'].unique()

array(['wii', 'nes', 'gb', 'ds', 'x360', 'ps3', 'ps2', 'snes', 'gba',
       'ps4', '3ds', 'n64', 'ps', 'xb', 'pc', '2600', 'psp', 'xone',
       'wiiu', 'gc', 'gen', 'dc', 'psv', 'sat', 'scd', 'ws', 'ng', 'tg16',
       '3do', 'gg', 'pcfx'], dtype=object)

In [37]:
# Получу список уникальных значений в категориальных данных в столбце rating
unique_df_rating = df['rating'].unique()
# Выведу список уникальных значений рейтинга ESRB
unique_df_rating

array(['E', 'Unknown', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'],
      dtype=object)

In [38]:
# Преобразую все значения в столбце rating к верхнему регистру
df['rating'] = df['rating'].str.upper()
# Выведу измененный список уникальных значений рейтинга ESRB
df['rating'].unique()

array(['E', 'UNKNOWN', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'],
      dtype=object)

In [39]:
# Получу список уникальных значений в категориальных данных в столбце year_of_release
unique_df_year = df['year_of_release'].unique()
# Выведу список уникальных годов выпуска игр
unique_df_year

array([2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010,
       2013, 2004, 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014,
       1992, 1997, 1993, 1994, 1982, 2016, 2003, 1986, 2000, 1995, 1991,
       1981, 1987, 1980, 1983])

In [40]:
# Считаю кол-во явных дубликатов в df
duplicated_rows = f"Количество дубликатов: {duplicated_rows}"

'Количество дубликатов: 239'

In [41]:
# Сортирую df по всем столбцам
df_sorted = df.sort_values(by=list(df.columns))

In [42]:
# Удалю дубликаты во всем df
df_no_duplicates = df_sorted.drop_duplicates()

In [43]:
# Сохраню и выведу количество строк после удаления дубликатов
final_count = df_no_duplicates.shape[0]
f'Количество строк после удаления дубликатов: {final_count}'

'Количество строк после удаления дубликатов: 16567'

In [44]:
# Проверю итоговый df на наличие пропусков
df.isna().sum()

name               0
platform           0
year_of_release    0
genre              0
na_sales           0
eu_sales           0
jp_sales           0
other_sales        0
critic_score       0
user_score         0
rating             0
dtype: int64

In [45]:
# Выведу итоговую информацию по предобработанному df с выводом первых 5 строк
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16806 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16806 non-null  object 
 1   platform         16806 non-null  object 
 2   year_of_release  16806 non-null  int64  
 3   genre            16806 non-null  object 
 4   na_sales         16806 non-null  float64
 5   eu_sales         16806 non-null  float64
 6   jp_sales         16806 non-null  float64
 7   other_sales      16806 non-null  float64
 8   critic_score     16806 non-null  float64
 9   user_score       16806 non-null  float64
 10  rating           16806 non-null  object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.5+ MB


,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E
1,Super Mario Bros.,nes,1985,platform,29.08,3.58,6.81,0.77,-1.0,-1.0,UNKNOWN
2,Mario Kart Wii,wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E
4,Pokemon Red/Pokemon Blue,gb,1996,role-playing,11.27,8.89,10.22,1.00,-1.0,-1.0,UNKNOWN


После нормализации текста найдено 239 явных дубликатов, они удалены. В данных осталось 16 567 строк, пропусков нет.

In [46]:
# Посчитаю кол-во удаленных строк в абсолютном значении
deleted_rows = initial_count - final_count
f'Количество удалённых строк в абсолютном значении: {deleted_rows}'

'Количество удалённых строк в абсолютном значении: 389'

In [47]:
# Посчитаю кол-во удаленных строк в относительном значении
percent_deleted = round((deleted_rows / initial_count)*100, 2)
f'Количество удалённых строк в относительном значении: {percent_deleted}%'

'Количество удалённых строк в относительном значении: 2.29%'

In [48]:
# Контроль: сколько строк удалено за всю предобработку
a, b = len(tmp), len(df_no_duplicates)
print(" Было строк в исходном датасете", a,
      '\n', "Осталось строк в датасете после обработки", b,
      '\n', "Удалено строк в датасете после обработки", a-b,
      '\n', "Процент потерь", round((a-b)/a*100, 2))

 Было строк в исходном датасете 16956 
 Осталось строк в датасете после обработки 16567 
 Удалено строк в датасете после обработки 389 
 Процент потерь 2.29


**Выводы по предобработке**

1. **Названия столбцов:** все 11 столбцов приведены к snake_case (например, `Year of Release` → `year_of_release`).
2. **Типы данных:** `eu_sales`, `jp_sales`, `user_score` переведены в `float64` (значения `unknown` и `tbd` стали `NaN`); `year_of_release` переведён в `int64`.
3. **Пропуски:** больше всего в `user_score` (9268, 54,66%), `critic_score` (8714, 51,39%) и `rating` (6871, 40,52%). Год заполнен по названию, продажи — средним по платформе и году, оценки — индикатором -1, рейтинг — `UNKNOWN`; строки без названия, жанра или года удалены. Пропусков не осталось.
4. **Дубликаты:** неявные дубликаты в `genre` устранены приведением к нижнему регистру (24 → 12 уникальных значений), `platform` приведён к нижнему регистру, `rating` — к верхнему; удалено 239 явных дубликатов.
5. **Итог:** из 16 956 строк осталось 16 567, удалено 389 строк (2,29%): 239 дубликатов и 150 строк с пропусками в названии, жанре или годе.

---

## 3. Срез данных за 2000–2013 годы

In [49]:
# Применю фильтр по годам выпуска игр, где интересует период с 2000 по 2013 год включительно
df_no_duplicates[(df_no_duplicates['year_of_release'] >= 2000) & (df_no_duplicates['year_of_release'] <= 2013)]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
3394,Frozen: Olaf's Quest,3ds,2013,platform,0.27,0.27,0.00,0.05,-1.0,-1.0,UNKNOWN
3906,Frozen: Olaf's Quest,ds,2013,platform,0.21,0.26,0.00,0.04,-1.0,-1.0,UNKNOWN
2478,Tales of Xillia 2,ps3,2012,role-playing,0.20,0.12,0.45,0.07,71.0,7.9,T
8460,.hack//G.U. Vol.1//Rebirth,ps2,2006,role-playing,0.00,0.00,0.17,0.00,-1.0,-1.0,UNKNOWN
7182,.hack//G.U. Vol.2//Reminisce,ps2,2006,role-playing,0.11,0.09,0.00,0.03,-1.0,-1.0,UNKNOWN
...,...,...,...,...,...,...,...,...,...,...,...
16731,thinkSMART: Chess for Kids,ds,2011,misc,0.01,0.00,0.00,0.00,-1.0,-1.0,E
647,uDraw Studio,wii,2010,misc,1.65,0.57,0.00,0.20,71.0,-1.0,E
8397,uDraw Studio: Instant Artist,wii,2011,misc,0.06,0.09,0.00,0.02,-1.0,-1.0,E
15836,uDraw Studio: Instant Artist,x360,2011,misc,0.01,0.01,0.00,0.00,54.0,5.7,E


In [50]:
# Сохраню новый срез данных в отдельно df_actual
df_actual = df_no_duplicates[(df_no_duplicates['year_of_release'] >= 2000) & (df_no_duplicates['year_of_release'] <= 2013)]

In [51]:
# Сделаю явную копию df перед изменениями столбцов
df_actual = df_actual.copy()

In [53]:
# Число игр в срезе 2000–2013
len(df_actual)

12900

---

## 4. Категоризация данных

In [54]:
# Создаю категорию для оценок пользователей: если значение -1, ставлю "нет оценки", иначе разбиваю на низкую, среднюю и высокую
df_actual['category_user_score'] = 'нет оценки'

mask = df_actual['user_score'] != -1
df_actual.loc[mask, 'category_user_score'] = pd.cut(df_actual.loc[mask, 'user_score'], bins=[0, 3, 8, 10], labels=['низкая оценка', 'средняя оценка', 'высокая оценка'], right=False, include_lowest=True)

In [55]:
# Проверю преобразованные уникальные значения категорий пользовательских оценок
df_actual['category_user_score'].unique()

array(['нет оценки', 'средняя оценка', 'высокая оценка', 'низкая оценка'],
      dtype=object)

In [56]:
# Создаю категорию для оценок критиков: если значение -1, ставлю "нет оценки", иначе разбиваю на низкую, среднюю и высокую
df_actual['category_critic_score'] = 'нет оценки'

mask = df_actual['critic_score'] != -1
df_actual.loc[mask, 'category_critic_score'] = pd.cut(df_actual.loc[mask, 'critic_score'], bins=[0, 30, 80, 100], labels=['низкая оценка', 'средняя оценка', 'высокая оценка'], right=False, include_lowest=True)

In [57]:
# Проверю преобразованные уникальные значения категорий оценок критиков
df_actual['category_critic_score'].unique()

array(['нет оценки', 'средняя оценка', 'высокая оценка', 'низкая оценка'],
      dtype=object)

In [58]:
# Проверяем, что граничные значения попали в нужные категории
tmp0, u_s, c_s = df_actual.copy(), 'user_score', 'critic_score'
cat_u, cat_c = 'category_user_score', 'category_critic_score'
print("Оценка 8 должна относиться категории 'высокая оценка'", "здесь это",
      tmp0[tmp0[u_s] == 8][cat_u].unique())
print("Оценка 3 должна относиться категории 'средняя оценка'", "здесь это",
      tmp0[tmp0[u_s] == 3][cat_u].unique())
print("Оценка 0 должна относиться категории 'низкая оценка'", "здесь это",
      tmp0[tmp0[u_s] == 0][cat_u].unique())
print("=================================")
print("Оценка 80 должна относиться категории 'высокая оценка'", "здесь это",
      tmp0[tmp0[c_s] == 80][cat_c].unique())
print("Оценка 30 должна относиться категории 'средняя оценка'", "здесь это",
      tmp0[tmp0[c_s] == 30][cat_c].unique())
print("Оценка 0 должна относиться категории 'низкая оценка'", "здесь это",
      tmp0[tmp0[c_s] == 0][cat_c].unique())

Оценка 8 должна относиться категории 'высокая оценка' здесь это ['высокая оценка']
Оценка 3 должна относиться категории 'средняя оценка' здесь это ['средняя оценка']
Оценка 0 должна относиться категории 'низкая оценка' здесь это ['низкая оценка']
Оценка 80 должна относиться категории 'высокая оценка' здесь это ['высокая оценка']
Оценка 30 должна относиться категории 'средняя оценка' здесь это ['средняя оценка']
Оценка 0 должна относиться категории 'низкая оценка' здесь это []


In [59]:
# Число игр в каждой категории пользовательских оценок
grouped_user_score = df_actual.groupby(['category_user_score'])['name'].count().sort_values(ascending=True)

In [60]:
# Выведу сгруппированные данные по пользовательским оценкам, отсортированные по возрастанию
grouped_user_score

category_user_score
низкая оценка      118
высокая оценка    2308
средняя оценка    4133
нет оценки        6341
Name: name, dtype: int64

In [61]:
# Число игр в каждой категории оценок критиков
grouped_critic_score = df_actual.groupby(['category_critic_score'])['name'].count().sort_values(ascending=True)

In [62]:
# Выведу сгруппированные данные по оценкам критиков, отсортированные по возрастанию
grouped_critic_score

category_critic_score
низкая оценка       57
высокая оценка    1712
средняя оценка    5491
нет оценки        5640
Name: name, dtype: int64

In [63]:
# Посчитаю количество игр для каждой платформы и отсортирую по убыванию
top_platforms = (df_actual.groupby('platform')['name'].count().sort_values(ascending=False))

In [64]:
# Выведу топ-7 платформ по количеству выпущенных игр
top_platforms.head(7)

platform
ps2     2139
ds      2128
wii     1290
psp     1190
x360    1139
ps3     1100
xb       817
Name: name, dtype: int64

## Итоговый вывод

1. **Цель:** подготовить данные о видеоиграх для анализа: исправить структуру, обработать пропуски, дубликаты и ошибки, сформировать срез за 2000–2013 годы.
2. **Исходные данные:** 16 956 строк и 11 столбцов; найдены строковые числа, значения `unknown` и `tbd`, пропуски и дубликаты. Больше всего пропусков в `user_score` (9268), `critic_score` (8714) и `rating` (6871).
3. **Обработка:** столбцы приведены к snake_case, `eu_sales`, `jp_sales`, `user_score` переведены в числа, `year_of_release` — в `int64`; значения `genre`, `platform` и `rating` нормализованы; пропуски в оценках заменены на -1, в рейтинге — на `UNKNOWN`.
4. **Удалено:** 389 строк (2,29%): 239 явных дубликатов и 150 строк без названия, жанра или года. Итоговый набор — 16 567 строк без пропусков.
5. **Срез 2000–2013:** `df_actual`, 12 900 игр.
6. **Категории оценок в срезе:**
   - пользователи: высокая — 2308, средняя — 4133, низкая — 118, нет оценки — 6341;
   - критики: высокая — 1712, средняя — 5491, низкая — 57, нет оценки — 5640.
7. **Топ-7 платформ по числу игр:** PS2 — 2139, DS — 2128, Wii — 1290, PSP — 1190, X360 — 1139, PS3 — 1100, Xbox — 817.

**Ограничения:** у 44–49% игр среза нет оценок критиков или пользователей (заполнены -1), а исходно у 40% игр не было рейтинга ESRB.